# Record linkage

In [2]:
import recordlinkage
import pandas as pd

In [ ]:
%pip install recordlinkage

In [3]:
hospital_accounts = pd.read_csv('hospital_account_info.csv', index_col='Account_Num')
hospital_reinbursement = pd.read_csv('hospital_reimbursement.csv', index_col='Provider_Num')

In [4]:
indexer = recordlinkage.Index()
# full means that all potential pairs are evaluated during the match
indexer.full()

<Index>

In [5]:
# candidates are all the pairs that we have to check
candidates = indexer.index(hospital_accounts, hospital_reinbursement)
print(len(candidates))

14399283


In [6]:
# we define the way we want to compare our tables
compare = recordlinkage.Compare()
compare.exact('City', 'Provider City', label='City')
compare.string('Facility Name', 'Provider Name', threshold=0.85, label='Hosp_Name')
compare.string('Address', 'Provider Street Address', method='jarowinkler', threshold=0.85, label='Hosp_Address')
features = compare.compute(candidates, hospital_accounts, hospital_reinbursement)

Takes few minutes to do the computations. We can try to speed this up a little bit. For instance it would be great to only compare hospitals that are in the same states.

In [7]:
indexer = recordlinkage.Index()
indexer.block(left_on='State', right_on='Provider State')
candidates = indexer.index(hospital_accounts, hospital_reinbursement)
print(len(candidates))

475830


The number of comparisons really dropped.

In [9]:
compare = recordlinkage.Compare()
compare.exact('City', 'Provider City', label='City')
compare.string('Facility Name', 'Provider Name', threshold=0.85, label='Hosp_Name')
compare.string('Address', 'Provider Street Address', method='jarowinkler', threshold=0.85, label='Hosp_Address')
features = compare.compute(candidates, hospital_accounts, hospital_reinbursement)

Now it takes only 10 seconds !

But what if there are some spelling mistakes in the state names (for instance "Tenessee" and "Tennessee") ? Some comparisons are not done because of these mistakes. To solve this issue, we can use "sorted neighborhood".

In [10]:
indexer = recordlinkage.Index()
indexer.sortedneighbourhood(left_on='State', right_on='Provider State')
candidates = indexer.index(hospital_accounts, hospital_reinbursement)
print(len(candidates))

998860


In [11]:
compare = recordlinkage.Compare()
compare.exact('City', 'Provider City', label='City')
compare.string('Facility Name', 'Provider Name', threshold=0.85, label='Hosp_Name')
compare.string('Address', 'Provider Street Address', method='jarowinkler', threshold=0.85, label='Hosp_Address')
features = compare.compute(candidates, hospital_accounts, hospital_reinbursement)

In [12]:
features

City  Hosp_Name  Hosp_Address
Account_Num Provider_Num                               
10605       537184           0        0.0           0.0
            803181           0        0.0           0.0
            450616           0        0.0           0.0
            854377           0        0.0           0.0
            560361           0        0.0           0.0
...                        ...        ...           ...
70226       815904           0        0.0           0.0
            746090           0        0.0           0.0
            193062           0        0.0           0.0
            834984           0        0.0           0.0
            365095           0        0.0           0.0

[998860 rows x 3 columns]

In the result, each row correspond to the comparisonw. If we have a 1, it means it's a match. 

In [13]:
features.sum(axis=1).value_counts().sort_index(ascending=False)

3.0      2290
2.0       461
1.0      8261
0.0    987848
dtype: int64

There are 988,187 rows with no matching values whatsoever. 7937 rows have at least one match, 451 have 2 and 2285 have 3 matches.

In [14]:
potential_matches = features[features.sum(axis=1)>1].reset_index()
potential_matches['Score'] = potential_matches.loc[:, 'City':'Hosp_Address'].sum(axis=1)
potential_matches

,Account_Num,Provider_Num,City,Hosp_Name,Hosp_Address,Score
0,51216,268781,0,1.0,1.0,2.0
1,55272,556917,1,1.0,1.0,3.0
2,87807,854637,1,1.0,1.0,3.0
3,51151,783146,1,0.0,1.0,2.0
4,11740,260374,1,1.0,1.0,3.0
...,...,...,...,...,...,...
2746,73846,138344,1,1.0,1.0,3.0
2747,82208,114556,1,1.0,1.0,3.0
2748,37566,286798,1,1.0,1.0,3.0
2749,64911,363945,1,1.0,1.0,3.0


In [15]:
hospital_accounts.loc[51216,:]

Facility Name             ST FRANCIS MEDICAL CENTER
Address                       2400 ST FRANCIS DRIVE
City                                   BRECKENRIDGE
State                                            MN
ZIP Code                                      56520
County Name                                  WILKIN
Phone Number                         (218) 643-3000
Hospital Type             Critical Access Hospitals
Hospital Ownership    Voluntary non-profit - Church
Name: 51216, dtype: object

In [16]:
hospital_reinbursement.loc[268781,:]

Provider Name                SAINT FRANCIS MEDICAL CENTER
Provider Street Address                 211 ST FRANCIS DR
Provider City                              CAPE GIRARDEAU
Provider State                                         MO
Provider Zip Code                                   63703
Total Discharges                                      141
Average Covered Charges                          42515.09
Average Total Payments                            5902.24
Average Medicare Payments                         4993.43
Name: 268781, dtype: object

In [17]:
hospital_accounts['Acct_Name_Lookup'] = hospital_accounts[['Facility Name', 'Address', 'City', 'State']].apply(lambda x: '_'.join(x), axis=1)
hospital_accounts

,Facility Name,Address,City,State,ZIP Code,County Name,Phone Number,Hospital Type,Hospital Ownership,Acct_Name_Lookup
Account_Num,,,,,,,,,,
10605,SAGE MEMORIAL HOSPITAL,STATE ROUTE 264 SOUTH 191,GANADO,AZ,86505,APACHE,(928) 755-4541,Critical Access Hospitals,Voluntary non-profit - Private,SAGE MEMORIAL HOSPITAL_STATE ROUTE 264 SOUTH 1...
24250,WOODRIDGE BEHAVIORAL CENTER,600 NORTH 7TH STREET,WEST MEMPHIS,AR,72301,CRITTENDEN,(870) 394-4113,Psychiatric,Proprietary,WOODRIDGE BEHAVIORAL CENTER_600 NORTH 7TH STRE...
10341,DOUGLAS GARDENS HOSPITAL,5200 NE 2ND AVE,MIAMI,FL,33137,MIAMI-DADE,(305) 751-8626,Acute Care Hospitals,Voluntary non-profit - Private,DOUGLAS GARDENS HOSPITAL_5200 NE 2ND AVE_MIAMI_FL
81101,SUNCOAST BEHAVIORAL HEALTH CENTER,4480 51ST ST W,BRADENTON,FL,34210,MANATEE,(941) 792-2222,Psychiatric,Proprietary,SUNCOAST BEHAVIORAL HEALTH CENTER_4480 51ST ST...
39835,TREASURE VALLEY HOSPITAL,8800 WEST EMERALD STREET,BOISE,ID,83704,ADA,(208) 373-5000,Acute Care Hospitals,Proprietary,TREASURE VALLEY HOSPITAL_8800 WEST EMERALD STR...
...,...,...,...,...,...,...,...,...,...,...
92281,JEWISH HOSPITAL - SHELBYVILLE,727 HOSPITAL DRIVE,SHELBYVILLE,KY,40065,SHELBY,(502) 647-4300,Acute Care Hospitals,Voluntary non-profit - Private,JEWISH HOSPITAL - SHELBYVILLE_727 HOSPITAL DRI...
65248,DAYTON CHILDREN'S HOSPITAL,ONE CHILDRENS PLAZA,DAYTON,OH,45404,MONTGOMERY,(937) 641-3450,Childrens,Voluntary non-profit - Private,DAYTON CHILDREN'S HOSPITAL_ONE CHILDRENS PLAZA...
92377,NORTH TEXAS STATE HOSPITAL,6515 KEMP BLVD,WICHITA FALLS,TX,76308,WICHITA,(940) 692-1220,Psychiatric,Government - State,NORTH TEXAS STATE HOSPITAL_6515 KEMP BLVD_WICH...


In [18]:
hospital_reinbursement['Reimbursement_Name_Lookup'] = hospital_reinbursement[['Provider Name', 'Provider Street Address', 'Provider City', 'Provider State']].apply(lambda x: '_'.join(x), axis=1)
hospital_reinbursement

,Provider Name,Provider Street Address,Provider City,Provider State,Provider Zip Code,Total Discharges,Average Covered Charges,Average Total Payments,Average Medicare Payments,Reimbursement_Name_Lookup
Provider_Num,,,,,,,,,,
839987,SOUTHEAST ALABAMA MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,118,20855.61,5026.19,4115.52,SOUTHEAST ALABAMA MEDICAL CENTER_1108 ROSS CLA...
519118,MARSHALL MEDICAL CENTER SOUTH,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,43,13289.09,5413.63,4490.93,MARSHALL MEDICAL CENTER SOUTH_2505 U S HIGHWAY...
733073,ELIZA COFFEE MEMORIAL HOSPITAL,205 MARENGO STREET,FLORENCE,AL,35631,73,22261.60,4922.18,4021.79,ELIZA COFFEE MEMORIAL HOSPITAL_205 MARENGO STR...
201752,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,12,10901.33,5343.50,4284.17,MIZELL MEMORIAL HOSPITAL_702 N MAIN ST_OPP_AL
678488,ST VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,74,28117.95,5947.12,4819.53,ST VINCENT'S EAST_50 MEDICAL PARK EAST DRIVE_B...
...,...,...,...,...,...,...,...,...,...,...
403227,BAYLOR SCOTT & WHITE MEDICAL CENTER- COLLEGE S...,700 SCOTT & WHITE DRIVE,COLLEGE STATION,TX,77845,37,20837.57,6389.76,3927.62,BAYLOR SCOTT & WHITE MEDICAL CENTER- COLLEGE S...
797810,WALNUT HILL MEDICAL CENTER,7502 GREENVILLE AVENUE,DALLAS,TX,75231,52,32453.85,21927.08,4423.52,WALNUT HILL MEDICAL CENTER_7502 GREENVILLE AVE...
611900,"BAY AREA REGIONAL MEDICAL CENTER, LLC",200 BLOSSOM STREET,WEBSTER,TX,77598,31,33006.84,19222.26,4497.16,"BAY AREA REGIONAL MEDICAL CENTER, LLC_200 BLOS..."


In [19]:
account_lookup = hospital_accounts[['Acct_Name_Lookup']].reset_index()
account_lookup

,Account_Num,Acct_Name_Lookup
0,10605,SAGE MEMORIAL HOSPITAL_STATE ROUTE 264 SOUTH 1...
1,24250,WOODRIDGE BEHAVIORAL CENTER_600 NORTH 7TH STRE...
2,10341,DOUGLAS GARDENS HOSPITAL_5200 NE 2ND AVE_MIAMI_FL
3,81101,SUNCOAST BEHAVIORAL HEALTH CENTER_4480 51ST ST...
4,39835,TREASURE VALLEY HOSPITAL_8800 WEST EMERALD STR...
...,...,...
5334,92281,JEWISH HOSPITAL - SHELBYVILLE_727 HOSPITAL DRI...
5335,65248,DAYTON CHILDREN'S HOSPITAL_ONE CHILDRENS PLAZA...
5336,92377,NORTH TEXAS STATE HOSPITAL_6515 KEMP BLVD_WICH...
5337,71562,GULF BREEZE HOSPITAL_1110 GULF BREEZE PKWY_GUL...


In [20]:
reimbursement_lookup = hospital_reinbursement[['Reimbursement_Name_Lookup']].reset_index()
reimbursement_lookup

,Provider_Num,Reimbursement_Name_Lookup
0,839987,SOUTHEAST ALABAMA MEDICAL CENTER_1108 ROSS CLA...
1,519118,MARSHALL MEDICAL CENTER SOUTH_2505 U S HIGHWAY...
2,733073,ELIZA COFFEE MEMORIAL HOSPITAL_205 MARENGO STR...
3,201752,MIZELL MEMORIAL HOSPITAL_702 N MAIN ST_OPP_AL
4,678488,ST VINCENT'S EAST_50 MEDICAL PARK EAST DRIVE_B...
...,...,...
2692,403227,BAYLOR SCOTT & WHITE MEDICAL CENTER- COLLEGE S...
2693,797810,WALNUT HILL MEDICAL CENTER_7502 GREENVILLE AVE...
2694,611900,"BAY AREA REGIONAL MEDICAL CENTER, LLC_200 BLOS..."
2695,693781,"RESOLUTE HEALTH HOSPITAL_555 CREEKSIDE XING,_N..."


In [21]:
account_merge = potential_matches.merge(account_lookup, how='left')
account_merge

,Account_Num,Provider_Num,City,Hosp_Name,Hosp_Address,Score,Acct_Name_Lookup
0,51216,268781,0,1.0,1.0,2.0,ST FRANCIS MEDICAL CENTER_2400 ST FRANCIS DRIV...
1,55272,556917,1,1.0,1.0,3.0,SCOTTSDALE OSBORN MEDICAL CENTER_7400 EAST OSB...
2,87807,854637,1,1.0,1.0,3.0,ORO VALLEY HOSPITAL_1551 EAST TANGERINE ROAD_O...
3,51151,783146,1,0.0,1.0,2.0,"ST. LUKE'S BEHAVIORAL HOSPITAL, LP_1800 EAST V..."
4,11740,260374,1,1.0,1.0,3.0,SUMMIT HEALTHCARE REGIONAL MEDICAL CENTER_2200...
...,...,...,...,...,...,...,...
2746,73846,138344,1,1.0,1.0,3.0,WESTERLY HOSPITAL_25 WELLS STREET_WESTERLY_RI
2747,82208,114556,1,1.0,1.0,3.0,KENT COUNTY MEMORIAL HOSPITAL_455 TOLL GATE RD...
2748,37566,286798,1,1.0,1.0,3.0,ROGER WILLIAMS MEDICAL CENTER_825 CHALKSTONE A...
2749,64911,363945,1,1.0,1.0,3.0,SOUTH COUNTY HOSPITAL INC_100 KENYON AVE_WAKEF...


In [22]:
final_merge = account_merge.merge(reimbursement_lookup, how='left')
final_merge

,Account_Num,Provider_Num,City,Hosp_Name,Hosp_Address,Score,Acct_Name_Lookup,Reimbursement_Name_Lookup
0,51216,268781,0,1.0,1.0,2.0,ST FRANCIS MEDICAL CENTER_2400 ST FRANCIS DRIV...,SAINT FRANCIS MEDICAL CENTER_211 ST FRANCIS DR...
1,55272,556917,1,1.0,1.0,3.0,SCOTTSDALE OSBORN MEDICAL CENTER_7400 EAST OSB...,SCOTTSDALE OSBORN MEDICAL CENTER_7400 EAST OSB...
2,87807,854637,1,1.0,1.0,3.0,ORO VALLEY HOSPITAL_1551 EAST TANGERINE ROAD_O...,ORO VALLEY HOSPITAL_1551 EAST TANGERINE ROAD_O...
3,51151,783146,1,0.0,1.0,2.0,"ST. LUKE'S BEHAVIORAL HOSPITAL, LP_1800 EAST V...",ST LUKE'S MEDICAL CENTER_1800 EAST VAN BUREN S...
4,11740,260374,1,1.0,1.0,3.0,SUMMIT HEALTHCARE REGIONAL MEDICAL CENTER_2200...,SUMMIT HEALTHCARE REGIONAL MEDICAL CENTER_2200...
...,...,...,...,...,...,...,...,...
2746,73846,138344,1,1.0,1.0,3.0,WESTERLY HOSPITAL_25 WELLS STREET_WESTERLY_RI,WESTERLY HOSPITAL_25 WELLS STREET_WESTERLY_RI
2747,82208,114556,1,1.0,1.0,3.0,KENT COUNTY MEMORIAL HOSPITAL_455 TOLL GATE RD...,KENT COUNTY MEMORIAL HOSPITAL_455 TOLL GATE RD...
2748,37566,286798,1,1.0,1.0,3.0,ROGER WILLIAMS MEDICAL CENTER_825 CHALKSTONE A...,ROGER WILLIAMS MEDICAL CENTER_825 CHALKSTONE A...
2749,64911,363945,1,1.0,1.0,3.0,SOUTH COUNTY HOSPITAL INC_100 KENYON AVE_WAKEF...,SOUTH COUNTY HOSPITAL INC_100 KENYON AVE_WAKEF...


In [23]:
cols = ['Account_Num', 'Provider_Num', 'Score', 'Acct_Name_Lookup', 'Reimbursement_Name_Lookup']
final_merge[cols].sort_values(by=['Account_Num', 'Score'], ascending=False)

,Account_Num,Provider_Num,Score,Acct_Name_Lookup,Reimbursement_Name_Lookup
2675,94995,825914,3.0,CLAIBORNE MEDICAL CENTER_1850 OLD KNOXVILLE HI...,CLAIBORNE MEDICAL CENTER_1850 OLD KNOXVILLE HI...
1987,94953,819181,3.0,LAKE CHARLES MEMORIAL HOSPITAL_1701 OAK PARK B...,LAKE CHARLES MEMORIAL HOSPITAL_1701 OAK PARK B...
1045,94943,680596,3.0,VALLEY PRESBYTERIAN HOSPITAL_15107 VANOWEN ST_...,VALLEY PRESBYTERIAN HOSPITAL_15107 VANOWEN ST_...
2319,94923,403151,3.0,UNIVERSITY COLO HEALTH MEMORIAL HOSPITAL CENTR...,UNIVERSITY COLO HEALTH MEMORIAL HOSPITAL CENTR...
2526,94887,752284,2.0,NEW YORK-PRESBYTERIAN BROOKLYN METHODIST HOSPI...,NEW YORK METHODIST HOSPITAL_506 SIXTH STREET_B...
...,...,...,...,...,...
2093,10165,188247,3.0,UTAH VALLEY HOSPITAL_1034 NORTH 500 WEST_PROVO_UT,UTAH VALLEY HOSPITAL_1034 NORTH 500 WEST_PROVO_UT
1836,10090,212069,3.0,CANONSBURG GENERAL HOSPITAL_100 MEDICAL BOULEV...,CANONSBURG GENERAL HOSPITAL_100 MEDICAL BOULEV...
2438,10043,140535,3.0,BETH ISRAEL DEACONESS HOSPITAL - PLYMOUTH_275 ...,BETH ISRAEL DEACONESS HOSPITAL - PLYMOUTH_275 ...
1971,10020,210657,3.0,ST FRANCIS MEDICAL CENTER_309 JACKSON STREET_M...,ST FRANCIS MEDICAL CENTER_309 JACKSON STREET_M...
